# Gini coding warm-up: 45 minutes
**Shaami's practice notebook.** It follows the job spec: load a client extract, clean it, join it, sense-check it, build the tables that go in a pack, fit a PD model, then check it (Gini, calibration, PSI) and reconcile it against a "SAS" output.

**How to use it**
- Run the setup cell once. It creates fake, deliberately messy data.
- Each exercise cell has a `# TODO` and some `...` to replace. Comments starting `# HINT` tell you which function to reach for.
- Run the **CHECK** cell straight after each exercise. If it prints ✅ you're done; if it errors, read the message.
- Time budget is in each heading. If you're stuck for more than 3 minutes, open the solutions notebook (the job spec says *two hours then ask*; you've got 45 minutes, so make it 3).
- 💬 **Say it out loud** boxes are what you'd say to an interviewer while coding. Practise saying them.

| Section | Minutes |
|---|---|
| 1. Inspect | 5 |
| 2. Clean | 8 |
| 3. Join and reconcile | 7 |
| 4. Summary table for a pack | 6 |
| 5. WoE and IV | 6 |
| 6. Logistic regression and Gini | 7 |
| 7. Calibration and PSI | 6 |
| 8. Bonus: reconcile Python vs SAS | if time |

## 0. Setup: run this, don't edit it

In [79]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 5000

# --- a mortgage book "as of development" ---------------------------------
ltv = np.clip(rng.normal(65, 15, n), 10, 110).round(1)
arrears = rng.choice([0, 1, 2, 3], size=n, p=[0.85, 0.09, 0.04, 0.02])
logit = -4 + 0.035 * (ltv - 60) + 0.9 * arrears          # the "true" risk
default_flag = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)

loans = pd.DataFrame({
    "acct_id": np.arange(1, n + 1).astype(float),
    "product": rng.choice(["Fixed", "fixed", "FIXED ", " Tracker", "tracker", "SVR", "svr "], n),
    "ltv": ltv,
    "arrears_months": arrears,
    "open_date": pd.to_datetime("2015-01-01") + pd.to_timedelta(rng.integers(0, 3000, n), unit="D"),
    "default_flag": default_flag,
})
loans["open_date"] = loans["open_date"].dt.strftime("%d/%m/%Y")        # dates arrive as UK text
loans.loc[rng.choice(n, 150, replace=False), "ltv"] = np.nan             # some missing LTVs
loans.loc[[17, 2301], "acct_id"] = np.nan                                # two broken IDs
loans = pd.concat([loans, loans.sample(25, random_state=1)], ignore_index=True)  # duplicate rows

# --- bureau file from another system: IDs stored as TEXT, a few accounts missing ---
keep = rng.random(n) > 0.04
bureau = pd.DataFrame({
    "acct_id": np.arange(1, n + 1)[keep].astype(str),
    "bureau_score": rng.integers(300, 800, keep.sum()),
})

# --- a more recent snapshot for monitoring: LTVs have drifted up (house prices fell) ---
m = 3000
recent = pd.DataFrame({
    "ltv": np.clip(rng.normal(71, 16, m), 10, 120).round(1),
    "arrears_months": rng.choice([0, 1, 2, 3], size=m, p=[0.82, 0.10, 0.05, 0.03]),
})
print("Setup done:", loans.shape, bureau.shape, recent.shape)

Setup done: (5025, 6) (4788, 2) (3000, 2)


## 1. Inspect (5 min)
First thing with any client extract: **look before you touch.** Shape, types, missing values, odd categories.

In [80]:
# TODO 1.1: print the shape, the column types, and the first 5 rows of loans
# HINT: .shape  .dtypes  .head()
print(loans.shape)
print(loans.dtypes)
loans.head()


(5025, 6)
acct_id           float64
product            object
ltv               float64
arrears_months      int64
open_date          object
default_flag        int64
dtype: object


,acct_id,product,ltv,arrears_months,open_date,default_flag
0,1.0,Fixed,69.6,0,25/01/2021,0
1,2.0,Tracker,49.4,0,01/08/2016,0
2,3.0,fixed,76.3,0,24/02/2018,0
3,4.0,fixed,79.1,0,10/09/2021,0
4,5.0,fixed,35.7,0,10/01/2020,0


In [67]:
loans.head()

,acct_id,product,ltv,arrears_months,open_date,default_flag
0,1.0,Fixed,69.6,0,25/01/2021,0
1,2.0,Tracker,49.4,0,01/08/2016,0
2,3.0,fixed,76.3,0,24/02/2018,0
3,4.0,fixed,79.1,0,10/09/2021,0
4,5.0,fixed,35.7,0,10/01/2020,0


👀 Look at `acct_id`. Its type is **float64**, not int. Why? (Answer in the next exercise.)

In [68]:
# TODO 1.2: count missing values in every column
# HINT: .isna() gives True/False, .sum() counts the Trues per column
missing = loans.isna().sum()
missing

acct_id             2
product             0
ltv               151
arrears_months      0
open_date           0
default_flag        0
dtype: int64

In [69]:
# CHECK 1.2
assert missing["ltv"] == loans["ltv"].isna().sum() and missing["acct_id"] == 2
print("✅ 2 missing IDs - that one NaN is why the whole ID column became float")

✅ 2 missing IDs - that one NaN is why the whole ID column became float


In [70]:
# TODO 1.3: how many distinct spellings of product are there? Show each with its count.
# HINT: .value_counts()
loans['product'].value_counts()

product
Fixed       749
svr         748
tracker     732
fixed       719
 Tracker    701
SVR         696
FIXED       680
Name: count, dtype: int64

💬 **Say it out loud:** *"Before cleaning I profile the data: shape, types, missing counts and category spellings. Two things jump out: account ID is float because of missing IDs, and product has seven spellings of three products."*

## 2. Clean (8 min)
Do one thing per step, and **check the row count after each step** so nothing disappears unexplained.

In [71]:
# TODO 2.1: drop exact duplicate rows. Print rows before and after.
before = len(loans)
loans = loans.drop_duplicates()       # HINT: .drop_duplicates()
print("rows before:", before, "after:", len(loans), "removed:", before - len(loans))

rows before: 5025 after: 5000 removed: 25


In [72]:
# CHECK 2.1
assert len(loans) == 5000
print("✅ 25 duplicates gone")

✅ 25 duplicates gone


In [73]:
loans = loans.dropna(subset=["acct_id"])
loans["acct_id"] = loans["acct_id"].astype("Int64")
print(loans["acct_id"].dtype, len(loans))

Int64 4998


In [74]:
# CHECK 2.2
assert str(loans["acct_id"].dtype) == "Int64" and len(loans) == 4998
print("✅ IDs are whole numbers again; 2 broken records removed (in real life you'd log them and tell the client)")

✅ IDs are whole numbers again; 2 broken records removed (in real life you'd log them and tell the client)


In [81]:
loans

,acct_id,product,ltv,arrears_months,open_date,default_flag
0,1.0,Fixed,69.6,0,25/01/2021,0
1,2.0,Tracker,49.4,0,01/08/2016,0
2,3.0,fixed,76.3,0,24/02/2018,0
3,4.0,fixed,79.1,0,10/09/2021,0
4,5.0,fixed,35.7,0,10/01/2020,0
...,...,...,...,...,...,...
5020,2417.0,Fixed,53.8,0,12/12/2022,0
5021,3671.0,Fixed,72.3,1,23/06/2018,0
5022,4839.0,svr,83.0,0,09/01/2015,0
5023,3091.0,Tracker,83.6,2,17/11/2015,0


In [38]:
# TODO 2.2: drop rows with no acct_id, then make acct_id a whole-number type that allows missing
# HINT: .dropna(subset=[...])   then   .astype("Int64")   <- capital I = nullable integer
loans = loans.dropna(subset=["acct_id"])
loans["acct_id"] = loans["acct_id"].astype("Int64") 
print(loans["acct_id"].dtype, len(loans))

TypeError: int() argument must be a string, a bytes-like object or a number, not 'ellipsis'

In [82]:
# TODO 2.3: standardise product so there are only 3 values: FIXED, TRACKER, SVR
# HINT: string methods live under .str  ->  .str.strip()  .str.upper()   (you can chain them)
loans["product"] = loans["product"].str.strip().str.upper()
loans["product"].value_counts()

product
FIXED      2148
SVR        1444
TRACKER    1433
Name: count, dtype: int64

In [83]:
# CHECK 2.3
assert set(loans["product"]) == {"FIXED", "TRACKER", "SVR"}
print("✅ 3 clean categories")

✅ 3 clean categories


In [84]:
# TODO 2.4: turn open_date (text like "25/03/2017") into a real date
# HINT: pd.to_datetime(column, format="%d/%m/%Y")
# GOTCHA: without a format, pandas may read 03/04 as 4 March (US style). UK data = day first.
loans["open_date"] = pd.to_datetime(loans["open_date"], format="%d/%m/%Y")
loans["open_date"].min(), loans["open_date"].max()

(Timestamp('2015-01-01 00:00:00'), Timestamp('2023-03-19 00:00:00'))

In [85]:
# CHECK 2.4
assert pd.api.types.is_datetime64_any_dtype(loans["open_date"])
print("✅ dates parsed")

✅ dates parsed


In [97]:
# TODO 2.5: missing LTV. First add a flag column ltv_missing (1 if missing, else 0),
#           THEN fill missing ltv with the median ltv.
# WHY the flag first? Missingness can itself predict default. Fill first and you lose that info.
# HINT: .isna().astype(int)    .fillna(value)    .median()
loans["ltv_missing"] = loans["ltv"].isna()
loans["ltv"] = loans["ltv"].isna().astype(int)

In [98]:
loans['ltv_missing'].value_counts()

ltv_missing
False    5025
Name: count, dtype: int64

In [87]:
# CHECK 2.5
assert loans["ltv"].isna().sum() == 0 and loans["ltv_missing"].sum() == loans["ltv_missing"].sum() > 0
print("✅ no missing LTV;", loans["ltv_missing"].sum(), "flagged")

AssertionError: 

💬 **Say it out loud:** *"I dedupe, fix the key type, standardise categories, parse dates with an explicit format, and flag missing values before imputing, and I check row counts at every step."*

## 3. Join and reconcile (7 min)
The bureau file stores IDs as **text**. This is where the "101" vs "101.0" bug lives.

In [ ]:
# TODO 3.1: make bureau acct_id the same type as loans (Int64), then LEFT join bureau onto loans.
# HINT: pd.to_numeric(col).astype("Int64")
# HINT: .merge(other, on="acct_id", how="left", validate="one_to_one", indicator=True)
#       validate= makes pandas ERROR if an ID is duplicated (a silent row explosion otherwise)
#       indicator= adds a _merge column saying "both" or "left_only"
bureau["acct_id"] = ...
df = ...
print(len(loans), len(df))

In [ ]:
# TODO 3.2: reconcile. (a) assert the row count didn't change. (b) count matched vs unmatched.
# HINT: assert len(a) == len(b), "message"      df["_merge"].value_counts()
...
...

In [ ]:
# TODO 3.3: list the first 5 account IDs that have NO bureau record (an "anti-join")
# HINT: filter rows where df["_merge"] == "left_only"
unmatched = ...
unmatched["acct_id"].head()

In [ ]:
# CHECK 3
assert len(df) == 4998 and 150 < len(unmatched) < 260
print("✅ every account kept;", len(unmatched), "have no bureau score - flag them, don't drop them silently")
df = df.drop(columns="_merge")

💬 **Say it out loud:** *"Left join so no accounts drop, validate one-to-one so duplicates can't multiply rows, then reconcile the counts and report the unmatched accounts to the client rather than losing them."*

## 4. Summary table for a client pack (6 min)
The bread and butter: band a variable, group, aggregate.

In [ ]:
# TODO 4.1: create ltv_band with edges 0, 60, 80, 90, infinity -> labels "<60", "60-80", "80-90", "90+"
# HINT: pd.cut(col, bins=[0, 60, 80, 90, np.inf], labels=[...], right=False)
# GOTCHA: right=False means 60 goes in "60-80" (like SAS formats "60-<80"). pandas' default puts 60 in "<60".
df["ltv_band"] = ...
df["ltv_band"].value_counts().sort_index()

In [ ]:
# TODO 4.2: one row per band with: accounts, defaults, default_rate
# HINT: df.groupby("ltv_band", observed=True).agg(accounts=("default_flag", "size"),
#                                                  defaults=("default_flag", "sum"))
#       then add default_rate = defaults / accounts
tbl = ...
tbl["default_rate"] = ...
tbl

In [ ]:
# CHECK 4
assert tbl["accounts"].sum() == len(df) and tbl["default_rate"].is_monotonic_increasing
print("✅ totals tie out to the book, and default rate rises with LTV (a sense check: does it make business sense?)")

💬 **Say it out loud:** *"Two checks on any summary table: it ties out to the total, and the trend makes business sense. Here, higher LTV means higher default rate."*

## 5. WoE and IV (6 min)
Same bands. WoE = ln(%good / %bad) per band. IV = Σ (%good − %bad) × WoE.

Rule of thumb for IV: <0.02 useless, 0.02–0.1 weak, 0.1–0.3 medium, >0.3 strong (>0.5 check for leakage).

In [ ]:
# TODO 5.1: build the WoE table
# goods = accounts - defaults ;  pct_good = goods / total goods ;  pct_bad = defaults / total defaults
woe = tbl[["accounts", "defaults"]].copy()
woe["goods"] = ...
woe["pct_good"] = ...
woe["pct_bad"] = ...
woe["woe"] = ...                                          # HINT: np.log(...)
woe["iv_part"] = ...
iv = woe["iv_part"].sum()
print("IV =", round(iv, 3))
woe

In [ ]:
# CHECK 5
assert woe["woe"].is_monotonic_decreasing and iv > 0.1
print("✅ WoE falls as LTV rises (riskier bands have more bads), IV =", round(iv, 3))

💬 **Say it out loud:** *"WoE is positive where goods are over-represented, negative where bads are. It should move smoothly across bands; if it zig-zags, I'd merge bands. IV tells me how much the variable separates goods from bads on its own."*

## 6. Logistic regression and Gini (7 min)
We'll use **statsmodels** because it gives a SAS-style summary (coefficients, standard errors, p-values). sklearn is fine too, but it hides those.

In [ ]:
# TODO 6.1: fit default_flag on ltv and arrears_months
# HINT: import statsmodels.api as sm
#       X = sm.add_constant(df[["ltv", "arrears_months"]])   <- add_constant = the intercept
#       model = sm.Logit(df["default_flag"], X).fit()
import statsmodels.api as sm
X = ...
model = ...
print(model.summary())

👀 **Read the output like a reviewer:**
- **coef**: both should be **positive** (higher LTV, more arrears → more risk). That's the *sign check*.
- **P>|z|**: tiny = the effect is real, not luck.
- The true values I used to make the data were 0.035 and 0.9. How close did you get?

In [ ]:
# TODO 6.2: predicted PD for every account, then Gini = 2*AUC - 1
# HINT: model.predict(X)      from sklearn.metrics import roc_auc_score   roc_auc_score(actual, predicted)
from sklearn.metrics import roc_auc_score
df["pd"] = ...
gini = ...
print("Gini =", round(gini, 3))

In [ ]:
# CHECK 6
assert (model.params[["ltv", "arrears_months"]] > 0).all() and 0.3 < gini < 0.9
print("✅ signs make sense, Gini =", round(gini, 3))

💬 **Say it out loud:** *"Gini measures ranking: does the model put the defaulters at the risky end. It doesn't tell me whether the PD level is right; that's calibration, next."*

## 7. Calibration and PSI (6 min)

In [ ]:
# TODO 7.1: split accounts into 10 equal-sized PD groups (deciles) and compare
#           mean predicted PD vs actual default rate in each
# HINT: pd.qcut(df["pd"], 10, labels=False, duplicates="drop")
#       then groupby the decile and take the .mean() of "pd" and "default_flag"
df["decile"] = ...
calib = ...
calib

👀 The two columns should track each other closely. A model fitted on this data is calibrated on average by construction; the real test is on new data over time.

In [ ]:
# TODO 7.2: write a PSI function and compare the LTV distribution at development (df) vs recent
# PSI = sum over bins of (actual% - expected%) * ln(actual% / expected%)
# Steps:  1) make bin edges from the EXPECTED (development) data, e.g. its deciles
#         2) count the share of each dataset in each bin
#         3) apply the formula. Add a tiny number (1e-6) to shares to avoid log(0).
def psi(expected, actual, n_bins=10):
    edges = np.quantile(expected, np.linspace(0, 1, n_bins + 1))
    edges[0], edges[-1] = -np.inf, np.inf          # catch values outside the dev range
    e_share = ...                                   # HINT: np.histogram(expected, edges)[0] / len(expected)
    a_share = ...
    e_share, a_share = e_share + 1e-6, a_share + 1e-6
    return ...

ltv_psi = psi(df["ltv"], recent["ltv"])
print("LTV PSI =", round(ltv_psi, 3))

In [ ]:
# CHECK 7
status = "green" if ltv_psi < 0.10 else "amber" if ltv_psi < 0.25 else "red"
print("✅ LTV PSI =", round(ltv_psi, 3), "->", status.upper())
print("   Sanity test: PSI of a dataset against itself should be ~0 ->", round(psi(df["ltv"], df["ltv"]), 6))

💬 **Say it out loud:** *"PSI compares today's distribution with development, using bins fixed on the development data. Under 0.10 green, 0.10 to 0.25 amber, above 0.25 red. LTV has drifted up here, which fits house prices falling. That's a leading signal; Gini and calibration confirm later once defaults come through."*

## 8. Bonus: reconcile Python vs SAS (if time)
Pretend the bank's SAS produced its own PD for every account. Your job: prove they match, and find any that don't.

In [ ]:
# setup for the bonus: run it
sas = df[["acct_id", "pd"]].rename(columns={"pd": "pd_sas"}).copy()
sas["pd_sas"] = sas["pd_sas"] + rng.normal(0, 1e-12, len(sas))   # tiny floating-point noise: fine
sas.loc[sas.index[[10, 400]], "pd_sas"] += 0.002                  # two real breaks
sas = sas.sample(frac=1, random_state=3)                          # SAS output arrives in a different order

In [ ]:
# TODO 8.1: join on acct_id (NOT on row order!), then flag rows where the PDs differ by more than 1e-8
# HINT: np.isclose(a, b, rtol=0, atol=1e-8) returns True where they match
rec = ...
rec["match"] = ...
print(rec["match"].value_counts())
rec.loc[~rec["match"], ["acct_id", "pd", "pd_sas"]]

💬 **Say it out loud:** *"Join on the key, never on row order. Outer join so I'd see accounts on only one side. Compare with an agreed tolerance, because exact equality fails on floating-point noise. Then investigate every break individually."*

---
### Done. Quick self-score
Which sections needed the solutions? Those are your revision list for tomorrow. The most likely live-test asks are sections **2, 3 and 4**, so be quick on those.